# はじめに

このノートでは、NGBoostの数理についてまとめておく。NGBoostの論文とPythonライブラリは下記よりアクセスできる。

- [NGBoost: Natural Gradient Boosting for Probabilistic Prediction](https://arxiv.org/abs/1910.03225)
- [stanfordmlgroup/ngboost: Natural Gradient Boosting for Probabilistic Prediction](https://github.com/stanfordmlgroup/ngboost)

NGBoostはXGBoostやLightGBMとは予測する対象が異なる。XGBoostやLightGBMは単一の予測値を返す一方で、NGBoostは確率分布のパラメタを返す。例えば、正規分布の平均と分散のようなパラメタを予測する。また、名前のとおり自然勾配(Natural Gradient)をもとにパラメタを更新する点でも異なる。そのため、ここでは論文で紹介されている下記の数理の部分に注目する。

<div align='center'><img src='./algo.png' width='600'></div>

## Proper Scoring Rule(スコアリングルール)

NGBoostのアルゴリズムではスコアリングルールというものが使用されている。スコアリングルールとは、ある目的変数に対応する確率分布に対してスコアを割り当てる関数のことで、NGBoostではこれを反復して最小化することを目指す。

例えば、特徴量$X$と目的変数$Y$があったとして、$X$のもとで$P(Y=1|X)=0.6, P(Y=0|X)=0.4$という確率分布が得られたとする。このとき、与えられた$Y$の分布は、実際の目的変数を適切に表現しているのか？という疑問が浮かぶ。スコアリング関数は、確率分布が目的変数をよく表しているときに、最小の値を与える関数であるため、この問題に解決策を与えてくれる。

目的変数$Y$の確率分布$P$が与えられたとき、スコアリングルールとしてログ損失を次のように定義する。

$$
S(P, y) = - log P(y)
$$

$y = 1$のとき、

$$
\begin{aligned}
S(P, y) &= -log P(1) \\
&= -log P(Y=1|X) \\
&= -log 0.6 \\
&=0.51083
\end{aligned}
$$

である。確率分布の値をずらしていくことで、スコア関数の変化を観察すると、$P(Y=1 \mid X)$が1に近づにつれて下がり続けることがわかる。

| $P(Y=1 \mid X)$ | $P(Y=0 \mid X)$ | Actual $Y$ | $S(P,1) = -\log_e P(Y=1 \mid X)$ |
| --------------: | --------------: | ------: | -------------------------------: |
|             0.6 |             0.4 |       1 |                          0.51083 |
|             0.7 |             0.3 |       1 |                          0.35667 |
|             0.8 |             0.2 |       1 |                          0.22314 |
|             0.9 |             0.1 |       1 |                          0.10536 |
|            0.95 |            0.05 |       1 |                          0.05129 |
|            0.99 |            0.01 |       1 |                          0.00436 |

$P(Y=1 \mid X)=1$のとき$S(P,1)=0$となる。

次に、真の分布$Q$が下記の通りわかっているとする。

$$
\begin{aligned}
Q(Y=1∣X)=0.8 \\
Q(Y=0∣X)=0.2
\end{aligned}
$$

これに対して、提案分布$P$として

$$
\begin{aligned}
P(Y=1∣X)=0.6 \\
P(Y=0∣X)=0.4
\end{aligned}
$$

を計算したとする。真の分布$Q$に対して、真の分布$Q$自身を使ってログ損失でスコアリングした場合の期待損失を計算する。

$$
\begin{aligned}
E_{Y∼Q} [S(Q,y)] &= Q(Y=1∣X)(−log Q(1))+Q(Y=0∣X)(−log Q(0)) \\
&= −Q(Y=1∣X)log Q(Y=1∣X)−Q(Y=0∣X)log Q(Y=0∣X) \\
&= −0.8log 0.8−0.2log 0.2 \\
&= 0.50040 \\
​\end{aligned}
$$

次に、真の分布$Q$に従って$Y$が発生しているにもかかわらず、提案分布として$P$を使った場合の期待損失を計算する。

$$
\begin{aligned}
E_{Y∼Q}[S(P,y)] &= Q(Y=1∣X)(−log P(1))+Q(Y=0∣X)(−log P(0)) \\
&= Q(Y=1∣X)(−logP(Y=1∣X))+Q(Y=0∣X)(−logP(Y=0∣X)) \\
&= −0.8log 0.6−0.2log0.4 \\
&= 0.59192 \\
​\end{aligned}
$$

これにより

$$
E_{Y∼Q} [S(Q,y)] \le E_{Y∼Q} [S(P,y)] 
$$

という関係が成り立つことがわかる。左辺は、確率変数$Y$が真の分布に従っているとき、真の分布$Q$に対応するスコアリングルールを使った場合の期待損失で、右辺は$Y$は真の分布$Q$に従っているのに、真の分布$Q$とは異なる別の提案分布$P$を使った場合の期待スコアリングルールを使った場合の期待損失。

このように、ある損失関数、つまりスコアリングルール$S(⋅,y)$について、真の分布を使ったときに期待損失が最も小さくなるなら、そのような損失関数を proper scoring ruleと呼ぶ。また、真の分布$Q$と予測分布$P$のズレを表す量を考えることができる。

$$
d(P,Q) = E_{Y∼Q} [S(P,y)] - E_{Y∼Q} [S(Q,y)]
$$

具体的に計算すると

$$
\begin{aligned}
d(P,Q) &= −0.8log0.6−0.2log0.4+0.8log0.8+0.2log0.2 \\
&= 0.8[log0.8−log0.6]+0.2[log0.2−log0.4] \\
&= 0.8log\frac{0.6}{0.8}​+0.2 log\frac{0.4}{​0.2}​
\end{aligned}
$$

となる。これは次のように書ける。

$$
\begin{aligned}
d(P,Q) &= q_{1} log \frac{q_{1}}{p_{1}} + q_{0} log \frac{q_{0}}{p_{0}} 
\end{aligned}
$$

とかける。これはKLダイバージェンスであり、ログ損失のスコアリング関数を使うと、KLダイバージェンスが得られる。そして、

$$
d(P,Q) \ge 0
$$

であるならば、

$$
E_{Y∼Q} [S(Q,y)] \le E_{Y∼Q} [S(P,y)] 
$$

となり、「提案分布$P$を使ったときの期待損失」が、「真の分布$Q$を使ったときの期待損失」より常に大きい、または同じになるなら、そのスコアリングルールは proper scoring rule である。つまり、真の分布$Q$をそのまま予測したときが、期待損失として一番良い、といえる。これがproper scoring ruleである。

NGBoostでは、$Y \sim P_{\theta}$として、目的変数の確率分布を返す。回帰であれば$Y \sim N(\mu, \sigma^2)$として、$\mu, \sigma^2$を学習する。このとき使う損失関数が proper scoring rule でないと、モデルは「真の分布に近づく」方向に学習しているとはいえない。損失関数が proper scoring rule なら、モデルは学習を通じて$P_{\theta}​≈Q$、提案分布$P_{\theta}$が真の分布$Q$に近づくように訓練される。一方で、損失関数が proper scoring rule ではないのであれば、$P_{\theta}​≈Q$という保証がなく、真の分布とは異なる分布でも良い分布と評価されかねない。

スコアリングルールによって定義されるズレ$d(P,Q)$が常に0以上なら、真の分布$Q$を予測したときに期待損失が最小になる。だからそのスコアリングルールは、確率分布を学習するために正しい損失関数だと言える。もしくは、本当の分布をそのまま予測したときに、平均的な損失が一番小さくなる、という性質を持っているのが proper scoring rule。

## 1. NGBoost の目的

通常の回帰モデルは、特徴量 $x$ に対して 1 つの点予測を返す。

$$
\hat{y}=f(x)
$$

一方、NGBoost は点予測ではなく、条件付き分布を予測する。

$$
Y \mid X=x \sim P_{\theta(x)}
$$

正規分布を仮定する場合は、次のように書ける。

$$
Y \mid X=x \sim \mathcal{N}\left(\mu(x),\sigma(x)^2\right)
$$

このとき NGBoost が学習するのは、$\mu(x), \sigma(x)$の両方を学習する。つまり、NGBoost は

$$
\hat{y}=0.01
$$

のような点予測ではなく、たとえば

$$
Y \mid x \sim \mathcal{N}(0.01,0.05^2)
$$

のような予測分布を返す。

## 2. 分布パラメータ

正規分布では、通常は$\mu, \sigma$をパラメータとして考える。ただし、標準偏差は必ず正でなければならない($\sigma>0$)。そのため、NGBoost では$s=\log\sigma$を使う。このとき、$\sigma=e^s$である。任意の実数 $s$ に対して、$e^s>0$なので、標準偏差が正であることが保証される。

したがって、正規分布のパラメータを

$$
\theta=
\begin{pmatrix}
\mu\\
s
\end{pmatrix}
$$

と置く。

ここで、

$$
s=\log\sigma
$$

である。

## 3. NGBoost のアルゴリズム

論文のアルゴリズムの用語を整理しておく。

### 入力

- データセット：$\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{n}$
- ブースティング回数：$M$
- 学習率：$\eta$
- 分布族：$P_{\theta}$
- スコア関数：$\mathcal{S}$
- ベースラーナー：$f$

### アルゴリズム本体

まず、初期分布パラメータを決める。

$$
\theta^{(0)}
\leftarrow
\arg\min_{\theta}
\sum_{i=1}^{n}
\mathcal{S}(\theta,y_i)
$$

これは、特徴量 $x_i$ を使わず、全データに共通する分布を 1 つ作るステップである。データ全体の平均など。その後、$m=1,\ldots,M$ について、次を繰り返す。各データ点 $i$ について、自然勾配を計算する。

$$
g_i^{(m)}
\leftarrow
\mathcal{I}_{\mathcal{S}}
\left(\theta_i^{(m-1)}\right)^{-1}
\nabla_{\theta}
\mathcal{S}\left(\theta_i^{(m-1)},y_i\right)
$$

次に、自然勾配を目的変数として、ベースラーナーを学習する。

$$
f^{(m)}
\leftarrow
\text{fit}
\left(
\{x_i,g_i^{(m)}\}_{i=1}^{n}
\right)
$$

次に、木の出力をどれくらいの大きさで使うかを決める。

$$
\rho^{(m)}
\leftarrow
\arg\min_{\rho}
\sum_{i=1}^{n}
\mathcal{S}
\left(
\theta_i^{(m-1)}
-
\rho \cdot f^{(m)}(x_i),
y_i
\right)
$$

最後に、分布パラメータを更新する。

$$
\theta_i^{(m)}
\leftarrow
\theta_i^{(m-1)}
-
\eta
\left(
\rho^{(m)}\cdot f^{(m)}(x_i)
\right)
$$

## 4. スコア関数：正規分布の負の対数尤度

正規分布の密度は、

$$
p(y\mid \mu,\sigma)
=
\frac{1}{\sqrt{2\pi}\sigma}
\exp\left(
-\frac{(y-\mu)^2}{2\sigma^2}
\right)
$$

である。NGBoost では、スコア関数として負の対数尤度を使うことが多い。

$$
\mathcal{S}(\mu,\sigma;y)
=
-\log p(y\mid \mu,\sigma)
$$

これを展開する。

$$
\mathcal{S}
=
-\log
\left[
\frac{1}{\sqrt{2\pi}\sigma}
\exp\left(
-\frac{(y-\mu)^2}{2\sigma^2}
\right)
\right]
$$

積の対数なので、

$$
\mathcal{S}
=
-\left[
\log\left(\frac{1}{\sqrt{2\pi}\sigma}\right)
+
\log\left\{
\exp\left(
-\frac{(y-\mu)^2}{2\sigma^2}
\right)
\right\}
\right]
$$

まず、

$$
\log\left(\frac{1}{\sqrt{2\pi}\sigma}\right)
=
-\log(\sqrt{2\pi})-\log\sigma
$$

また、

$$
\log\left\{
\exp\left(
-\frac{(y-\mu)^2}{2\sigma^2}
\right)
\right\}
=
-\frac{(y-\mu)^2}{2\sigma^2}
$$

したがって、

$$
\mathcal{S}
=
-\left[
-\log(\sqrt{2\pi})
-
\log\sigma
-
\frac{(y-\mu)^2}{2\sigma^2}
\right]
$$

よって、

$$
\mathcal{S}
=
\log(\sqrt{2\pi})
+
\log\sigma
+
\frac{(y-\mu)^2}{2\sigma^2}
$$

である。定数項

$$
\log(\sqrt{2\pi})
$$

を除くと、

$$
\mathcal{S}
=
\log\sigma
+
\frac{(y-\mu)^2}{2\sigma^2}
$$

となる。ここで、

$$
s=\log\sigma
$$

と置くと、

$$
\sigma=e^s
$$

であり、

$$
\sigma^2=e^{2s}
$$

なので、

$$
\mathcal{S}(\mu,s;y)
=
s+
\frac{(y-\mu)^2}{2e^{2s}}
$$

となる。この式を使って、NGBoost は

$$
\mu
$$

と

$$
s=\log\sigma
$$

を更新していく。

## 5. 具体的な数値例

ある 1 つのデータ点について考える。

実測値を、

$$
y=0.10
$$

とする。

現在の予測分布を、

$$
\mu=0.00, \sigma=0.05
$$

とする。したがって、

$$
s=\log(0.05)
$$

である。このとき標準化すると、

$$
\begin{aligned}
z &=\frac{y-\mu}{\sigma} \\
&=\frac{0.10-0.00}{0.05} \\
&=2
\end{aligned}
$$

なので、実測値は現在の予測平均から 2 標準偏差ぶん右にあることがわかる。

## 6. 普通の勾配の計算

スコア関数は、

$$
\mathcal{S}(\mu,s;y)
=
s+
\frac{(y-\mu)^2}{2e^{2s}}
$$

である。

まず、$\mu$ 方向の勾配を求める。

$$
\frac{\partial \mathcal{S}}{\partial \mu}
=
\frac{\partial}{\partial\mu}
\left[
s+
\frac{(y-\mu)^2}{2e^{2s}}
\right]
$$

$s$ は $\mu$ に依存しないので、

$$
\frac{\partial s}{\partial\mu}=0
$$

である。したがって、

$$
\begin{aligned}
\frac{\partial \mathcal{S}}{\partial \mu}
= \frac{\partial}{\partial\mu} \left[ \frac{(y-\mu)^2}{2e^{2s}} \right] \\
= \frac{1}{2e^{2s}} \frac{\partial}{\partial\mu}(y-\mu)^2
\end{aligned}
$$

ここで、

$$
\frac{\partial}{\partial\mu}(y-\mu)^2 = 2(y-\mu)(-1) = -2(y-\mu)
$$

だから、

$$
\begin{aligned}
\frac{\partial \mathcal{S}}{\partial \mu} &= \frac{1}{2e^{2s}}\{-2(y-\mu)\} \\
&= -\frac{y-\mu}{e^{2s}} \\
&= -\frac{y-\mu}{\sigma^2} \\
&= \frac{\mu-y}{\sigma^2}
\end{aligned}
$$

である。数値を代入すると、

$$
\begin{aligned}
\frac{\partial \mathcal{S}}{\partial \mu} &= \frac{0.00-0.10}{0.05^2} \\
&= \frac{-0.10}{0.0025} \\
&= -40
\end{aligned}
$$

である。次に、$s=\log\sigma$ 方向の勾配を求める。

$$
\frac{\partial \mathcal{S}}{\partial s}
=
\frac{\partial}{\partial s}
\left[
s+
\frac{(y-\mu)^2}{2e^{2s}}
\right]
$$

まず、

$$
\frac{\partial s}{\partial s}=1
$$

である。次に、

$$
\frac{(y-\mu)^2}{2e^{2s}}
=
\frac{(y-\mu)^2}{2}e^{-2s}
$$

なので、

$$
\begin{aligned}
\frac{\partial}{\partial s} \left[ \frac{(y-\mu)^2}{2}e^{-2s} \right] &= \frac{(y-\mu)^2}{2} \cdot (-2) e^{-2s} \\
&= - (y-\mu)^2 e^{-2s} \\
&= -\frac{(y-\mu)^2}{e^{2s}} \\
&= -\frac{(y-\mu)^2}{\sigma^2}
\end{aligned}
$$

したがって、

$$
\frac{\partial \mathcal{S}}{\partial s}
=
1-
\frac{(y-\mu)^2}{\sigma^2}
$$

である。数値を代入すると、

$$
\begin{aligned}
\frac{\partial \mathcal{S}}{\partial s} &= 1-\frac{(0.10-0.00)^2}{0.05^2} \\
&=1-\frac{0.01}{0.0025} \\
&=1-4 \\
&=-3
\end{aligned}
$$

である。したがって、普通の勾配ベクトルは、

$$
\nabla_{\theta}\mathcal{S}(\theta,y)
=
\begin{pmatrix}
-40\\
-3
\end{pmatrix}
$$

である。


## 7. 自然勾配を使う理由

普通の勾配では、$\mu$ 方向の勾配が$-40$と大きくなった。これは、

$$
\frac{\partial \mathcal{S}}{\partial \mu}
=
\frac{\mu-y}{\sigma^2}
$$

であり、$\sigma^2=0.0025$が小さいためである。しかし、標準偏差が小さい分布では、平均を少し動かすだけで、分布そのものが大きく変わる。この「分布そのものの変化量」を見る代表的な量が KL ダイバージェンスである。平均だけ違い、分散が同じ正規分布について、

$$
P=\mathcal{N}(\mu_1,\sigma^2) \\
Q=\mathcal{N}(\mu_2,\sigma^2)
$$

とすると、

$$
D_{\mathrm{KL}}(P\|Q) = \frac{(\mu_1-\mu_2)^2}{2\sigma^2}
$$

である。同じ平均移動$\Delta\mu=0.1$を考える。もし、$\sigma=1$なら、

$$
D_{\mathrm{KL}} = \frac{0.1^2}{2\times1^2} = \frac{0.01}{2} =0.005
$$

である。ほとんど同じ分布である。一方、$\sigma=0.01$なら、

$$
D_{\mathrm{KL}} = \frac{0.1^2}{2\times0.01^2} = \frac{0.01}{2\times0.0001} = \frac{0.01}{0.0002} =50
$$

である。

同じ $0.1$ の平均移動でも、標準偏差が小さいと、分布としては大きく変わってしまう。したがって、分布パラメータを更新するときは、単なるパラメータ座標上の勾配ではなく、分布空間の歪みを考慮した勾配を使う必要がある。これが自然勾配である。そして、この分布空間の歪みを教えてくれるのがフィッシャー情報量である。

## 8. フィッシャー情報行列

フィッシャー情報行列はパラメータが複数の場合に使われるフィッシャー情報量のようなもので、フィッシャー情報量は確率変数がパラメタに対して持っている情報の量を表す。フィッシャー情報量は下記がわかりやすい。

- [わかりやすいフィッシャー情報量 - Yosshi Labo.](https://yosshiblog.jp/%E3%83%95%E3%82%A3%E3%83%83%E3%82%B7%E3%83%A3%E3%83%BC%E6%83%85%E5%A0%B1%E9%87%8F/)

正規分布で、

$$
\theta=
\begin{pmatrix}
\mu\\
s
\end{pmatrix}
$$

$$
s=\log\sigma
$$

としたとき、フィッシャー情報行列は、

$$
\mathcal{I}_{\mathcal{S}}(\theta)
=
\begin{pmatrix}
1/\sigma^2 & 0\\
0 & 2
\end{pmatrix}
$$

である。今回、

$$
\sigma=0.05
$$

なので、

$$
\begin{aligned}
1/\sigma^2　&=\frac{1}{0.05^2} \\
&=\frac{1}{0.0025} \\
&=400
\end{aligned}
$$

である。したがって、

$$
\mathcal{I}_{\mathcal{S}}(\theta)
=
\begin{pmatrix}
400 & 0\\
0 & 2
\end{pmatrix}
$$

である。これは、

- $\mu$ 方向は 400 倍動きやすい
- $s$ 方向は 2 倍動きやすい

という意味である。

## 9. 自然勾配の計算

論文ののアルゴリズムでは、自然勾配を次のように計算している。

$$
g_i^{(m)}
=
\mathcal{I}_{\mathcal{S}}
\left(\theta_i^{(m-1)}\right)^{-1}
\nabla_{\theta}
\mathcal{S}
\left(\theta_i^{(m-1)},y_i\right)
$$

今回の例では、

$$
\mathcal{I}_{\mathcal{S}}(\theta)
=
\begin{pmatrix}
400 & 0\\
0 & 2
\end{pmatrix}
$$

なので、

$$
\mathcal{I}_{\mathcal{S}}(\theta)^{-1}
=
\begin{pmatrix}
1/400 & 0\\
0 & 1/2
\end{pmatrix}
=
\begin{pmatrix}
0.0025 & 0\\
0 & 0.5
\end{pmatrix}
$$

である。普通の勾配は、

$$
\nabla_{\theta}\mathcal{S}
=
\begin{pmatrix}
-40\\
-3
\end{pmatrix}
$$

だった。したがって、自然勾配は、

$$
g
=
\begin{pmatrix}
0.0025 & 0\\
0 & 0.5
\end{pmatrix}
\begin{pmatrix}
-40\\
-3
\end{pmatrix}
$$

である。1 行目は、

$$
0.0025\times(-40)+0\times(-3)
=-0.1
$$

2 行目は、

$$
0\times(-40)+0.5\times(-3)
=-1.5
$$

である。よって、

$$
g=
\begin{pmatrix}
-0.1\\
-1.5
\end{pmatrix}
$$

である。これが、アルゴリズムに出てくる自然勾配

$$
g_i^{(m)}
$$

である。


## 10. 自然勾配の解釈

自然勾配は

$$
g=
\begin{pmatrix}
-0.1\\
-1.5
\end{pmatrix}
$$

である。論文のアルゴリズムでは、後で

$$
\theta_i^{(m)}
\leftarrow
\theta_i^{(m-1)}
-
\eta
\left(
\rho^{(m)}\cdot f^{(m)}(x_i)
\right)
$$

として引く。したがって、自然勾配が

$$
\begin{pmatrix}
-0.1\\
-1.5
\end{pmatrix}
$$

なら、自然勾配がマイナスなので、更新方向はその反対になり、

$$
\begin{pmatrix}
0.1\\
1.5
\end{pmatrix}
$$

である。つまり、

- $\mu$ を増やす
- $s=\log\sigma$ も増やす

という更新になる。これは直感的にも自然である。今回、実測値は、

$$
y=0.10
$$

であり、現在の平均は、

$$
\mu=0.00
$$

だった。実測値は平均より上にあるので、平均を上げる必要がある。また、実測値は 2 標準偏差ぶん離れていたので、現在の分布は少し狭すぎる可能性がある。そのため、標準偏差も広げる方向に更新される。

## 11. ベースラーナーが学習しているもの

論文のアルゴリズムでは、ここでベースラーナーがでてくる。

$$
f^{(m)}
\leftarrow
\text{fit}
\left(
\{x_i,g_i^{(m)}\}_{i=1}^{n}
\right)
$$

ここが重要である。このベースラーナー $f^{(m)}$ は、目的変数 $y_i$ を予測しているのではない。また、$\mu_i$ や $\sigma_i$ を直接予測しているわけでもない。学習しているのは、

$$
x_i \mapsto g_i^{(m)}
$$

である。つまり、特徴量$x_i$のとき、現在の分布パラメータをどちらに修正すべきかを学習している。今回の 1 データ点では、

$$
g=
\begin{pmatrix}
-0.1\\
-1.5
\end{pmatrix}
$$

だった。したがって、理想的にはベースラーナーが、天下り的に下記を返すとすると、

$$
f^{(m)}(x)
=
\begin{pmatrix}
-0.1\\
-1.5
\end{pmatrix}
$$

を返すように学習する。

## 12. スケーリング係数 $\rho$

論文のアルゴリズムでは、次に

$$
\rho^{(m)}
\leftarrow
\arg\min_{\rho}
\sum_{i=1}^{n}
\mathcal{S}
\left(
\theta_i^{(m-1)}
-
\rho\cdot f^{(m)}(x_i),
y_i
\right)
$$

を計算する。これは、学習した木の出力をどれくらいの大きさで使うかを決めるステップである。木の出力

$$
f^{(m)}(x_i)
$$

は、更新方向を表している。しかし、その方向にどれくらい進むべきかは別問題である。そのため、

$$
\rho^{(m)}
$$

でスケールを調整する。要するに$\theta_i^{(m-1)}$は現在のパラメタで、$f^{(m)}(x_i)$は決定木が提案した更新方向。

$$
\theta_i^{(m-1)} - \rho\cdot f^{(m)}(x_i)
$$

これは、「$\rho$だけ進んだ後の仮の分布パラメータ」で、その仮の分布パラメータで損失を計算。

$$
\mathcal{S}
\left(
\theta_i^{(m-1)}
-
\rho\cdot f^{(m)}(x_i),
y_i
\right)
$$

これを全データで足します。

$$
\sum_{i=1}^{n}
\mathcal{S}
\left(
\theta_i^{(m-1)}
-
\rho\cdot f^{(m)}(x_i),
y_i
\right)
$$

そして、この合計損失が一番小さくなるスケーリング係数(ステップサイズ)を利用する。ここでは理解のための数値例では、簡単化して

$$
\rho^{(m)}=1
$$

とする。

## 13. 1 データ点での 1 イタレーション更新

学習率を$\eta=0.1$とする。簡単化のため、$\rho=1$とする。また、ベースラーナーが自然勾配を完全に再現できたとして、

$$
f^{(m)}(x)
=
\begin{pmatrix}
-0.1\\
-1.5
\end{pmatrix}
$$

とする。更新式は、

$$
\theta^{(1)}
=
\theta^{(0)}
-
\eta f^{(m)}(x)
$$

である。したがって、

$$
\begin{aligned}
\begin{pmatrix}
\mu^{(1)}\\
s^{(1)}
\end{pmatrix}
=
\begin{pmatrix}
\mu^{(0)}\\
s^{(0)}
\end{pmatrix}
-
0.1
\begin{pmatrix}
-0.1\\
-1.5
\end{pmatrix}
=
\begin{pmatrix}
\mu^{(0)}\\
s^{(0)}
\end{pmatrix}
+
\begin{pmatrix}
0.01\\
0.15
\end{pmatrix}
\end{aligned}
$$

である。したがって、

$$
\mu^{(1)} = 0.00+0.01 = 0.01 \\
s^{(1)} = \log(0.05)+0.15
$$

なので、標準偏差に戻すと、

$$
\begin{aligned}
\sigma^{(1)}
&= e^{s^{(1)}} \\
&= e^{\log(0.05)+0.15} \\
&= e^{\log(0.05)}e^{0.15} \\
&= 0.05e^{0.15} \\
\end{aligned}
$$

ここで、

$$
e^{0.15}\approx1.1618
$$

なので、

$$
\sigma^{(1)} \approx 0.05\times1.1618 \approx0.0581
$$

である。したがって、1 イタレーション後に、予測分布は

$$
\mathcal{N}(0.00,0.05^2)
$$

から、

$$
\mathcal{N}(0.01,0.0581^2)
$$

に更新されることになる。ここでは1レコードの1イタレーションの流れを追ったが、データ$n$件に増えても同じである。1回目のイテレーションで、1から$n$レコードまでの各自然勾配を計算し、弱学習器で特徴量から分布の修正方向を学習。そして、損失が小さくなるように、学習率、スケーリング係数をもとに各データのパラメタを更新する。

<div align='center'><img src='./algo.png' width='600'></div>



## まとめ
### ポイント 1：NGBoost は点ではなく分布を予測する

通常の回帰は$\hat{y}=f(x)$を出す。NGBoost は$Y\mid X=x \sim P_{\theta(x)}$を出す。正規分布なら$\mu(x),\sigma(x)$を予測する。

### ポイント 2：損失は負の対数尤度

正規分布なら$\mathcal{S}(\mu,s;y)=s+\frac{(y-\mu)^2}{2e^{2s}}$である。

この損失は、

- 平均を実測値に近づける
- 標準偏差を適切な広さにする

という 2 つの力を持つ。

### ポイント 3：普通の勾配ではなく自然勾配を使う

普通の勾配は、$\nabla_{\theta}\mathcal{S}$である。
自然勾配は、$g = \mathcal{I}_{\mathcal{S}}(\theta)^{-1} \nabla_{\theta}\mathcal{S}$である。
正規分布の場合、

$$
\mathcal{I}_{\mathcal{S}}(\theta)
=
\begin{pmatrix}
1/\sigma^2 & 0\\
0 & 2
\end{pmatrix}
$$

である。

### ポイント 4：フィッシャー情報行列は分布空間の歪みを補正する

同じ$\Delta\mu=0.1$でも、$\sigma=1$なら小さな変化であり、$\sigma=0.01$なら大きな変化である。

KL ダイバージェンスで見ると$D_{\mathrm{KL}}=\frac{(\Delta\mu)^2}{2\sigma^2}$なので、$\sigma=1$なら、$D_{\mathrm{KL}}=0.005$。一方、$\sigma=0.01$なら、$D_{\mathrm{KL}}=50$となる。自然勾配は、このような分布空間の歪みを補正している。

### ポイント 5：ベースラーナーは $y$ ではなく自然勾配を学習する

NGBoost の各イタレーションで学習する木は、$x_i \mapsto y_i$ではなく、$x_i \mapsto g_i^{(m)}$を学習する。

つまり、この特徴量では、分布パラメータをどちらに修正すべきかを学習している。


### ポイント 6：最後に分布パラメータを更新する

更新式は、

$$
\theta_i^{(m)}
=
\theta_i^{(m-1)}
-
\eta
\left(
\rho^{(m)}\cdot f^{(m)}(x_i)
\right)
$$

である。

正規分布なら、

$$
\theta_i^{(m)}
=
\begin{pmatrix}
\mu_i^{(m)}\\
s_i^{(m)}
\end{pmatrix}
$$

を更新する。

最後に、

$$
\sigma_i^{(m)}=e^{s_i^{(m)}}
$$

として標準偏差に戻す。

## 参考文献

- [NGBoostを読んで、実装する。 - nykergoto’s blog](https://nykergoto.hatenablog.jp/entry/2020/05/01/NGBoost%E3%82%92%E8%AA%AD%E3%82%93%E3%81%A7%E5%AE%9F%E8%A3%85%E3%81%99%E3%82%8B)
- [NGBoost Algorithm Explained with a Numerical Example | by DarkProgrammerPB | Medium](https://medium.com/@darkprogrammerpb/ngboost-algorithm-explained-with-a-numerical-example-586f9aa76af7)